In [1]:
import numpy as np
from Autograd import WhyyTorch as wt,cross_entropy_loss
import matplotlib.pyplot as plt

In [2]:
"""
What:Creating a Training dataset for the bigram neural network
Why: NN take numbers as input so to predict th next most probable character we need to convert the characters into numbers, and make pairs of characters to train the model on.
"""
words = open('bigram.txt').read().splitlines()
stoi = {'.': 0, **{chr(ord('a') + i): i + 1 for i in range(26)}} 
itos = {i: c for c, i in stoi.items()}
count = 0
print(f"Our Dictionary :{stoi}")
xs,ys = [],[]
for w in words:
        count += 1
print(count)
print(f"Total Words :{count}")

Our Dictionary :{'.': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26}
32033
Total Words :32033


In [3]:
#Building the dataset
block_size = 3
X,Y = [],[]
for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
        # print(''.join(itos[i] for i in context), '->', itos[ix])
        
X = wt(X)
Y = wt(Y)

In [4]:
X.shape,X.data.dtype, Y.shape,Y.data.dtype

((228146, 3), dtype('float32'), (228146,), dtype('float32'))

## Practise Bigram

In [28]:
np.random.seed(42)
C = wt(np.random.randn(27, 2))

In [45]:
X.data.shape

(32, 3)

In [68]:
emb = C.data[X.data.astype(int)]
emb_wt = wt(emb.reshape(-1, 6))
emb_wt.shape

(32, 6)

In [69]:
W1 = wt(np.random.randn(6, 100))
b1 = wt(np.random.randn(100))

In [71]:
h = wt.tanh(emb_wt @ W1 + b1)
h.shape

(32, 100)

In [73]:
W2 = wt(np.random.randn(100, 27))
b2 = wt(np.random.randn(27))

In [75]:
logits = h @ W2 + b2
logits.shape

(32, 27)

In [76]:
count = logits.exp()
probs = count / count.sum(axis=1, keepdims=True)

In [88]:
loss = -np.log(probs.data[np.arange(Y.data.shape[0]), Y.data.astype(int)]).mean()
print(loss)

18.328281


## Bigram with 3 letter context

In [32]:
np.random.seed(42)
# Setting Up the Paramters
# Network Looks like 27,10 -> 30,100 -> 100,27
# C is the embedding space 10d for each of the 27 characters
C = wt(np.random.randn(27, 10))
W1 = wt(np.random.randn(30, 200))
b1 = wt(np.random.randn(200))
W2 = wt(np.random.randn(200, 27))   
b2 = wt(np.random.randn(27))
parameters = [C, W1, b1, W2, b2]
print(f"Total Parameters :{sum(p.data.size for p in parameters)}")

Total Parameters :11897


In [33]:
# Train / Dev / Test split (80 / 10 / 10), WORD-LEVEL shuffle
# Shuffle at the word level (not row level) so that every row produced
# from a given word stays in the same split -> no context leakage.
np.random.seed(42)
shuffled_words = words.copy()
np.random.shuffle(shuffled_words)

n1 = int(0.8 * len(shuffled_words))
n2 = int(0.9 * len(shuffled_words))

def build_split(word_list):
    Xs, Ys = [], []
    for w in word_list:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            Xs.append(context)
            Ys.append(ix)
            context = context[1:] + [ix]
    return np.array(Xs, dtype=np.int64), np.array(Ys, dtype=np.int64)

Xtr,  Ytr  = build_split(shuffled_words[:n1])
Xdev, Ydev = build_split(shuffled_words[n1:n2])
Xte,  Yte  = build_split(shuffled_words[n2:])

print(f"train: {Xtr.shape}, dev: {Xdev.shape}, test: {Xte.shape}")

train: (182671, 3), dev: (22784, 3), test: (22691, 3)


In [ ]:
# Hyperparameters for the model and training
batch_size = 32
lr = 0.0001
for epoch in range(20000):
    
    # Mini batch from TRAIN split only
    ix = np.random.randint(0, Xtr.shape[0], (batch_size,))
    Xb = Xtr[ix]
    Yb = Ytr[ix]

    # Forward pass
    emb = C[Xb].reshape(Xb.shape[0], -1)   # (B, 30)
    h = wt.tanh(emb @ W1 + b1)             # (B, 200)
    logits = h @ W2 + b2                   # (B, 27)
    
    # Calculate loss
    loss = cross_entropy_loss(logits, Yb)
    
    # Optimizer zero grad
    for p in parameters:
        p.zero_grad()
        
    # Loss Backward
    loss.backward()
    
    # Optimizer step
    for p in parameters:
        p.data -= lr * p.grad
print("Final loss of last batch:", float(loss.data))

Final loss: 2.3867125511169434


In [35]:
# Evaluate on each split , lr=0.1
def split_loss(X_split, Y_split):
    emb = C[X_split].reshape(X_split.shape[0], -1)
    h = wt.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return float(cross_entropy_loss(logits, Y_split).data)

print(f"train loss: {split_loss(Xtr,  Ytr):.4f}")
print(f"dev   loss: {split_loss(Xdev, Ydev):.4f}")
print(f"test  loss: {split_loss(Xte,  Yte):.4f}")

train loss: 2.5418
dev   loss: 2.5470
test  loss: 2.5739


In [ ]:
# Evaluate on each split , lr=0.01
def split_loss(X_split, Y_split):
    emb = C[X_split].reshape(X_split.shape[0], -1)
    h = wt.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return float(cross_entropy_loss(logits, Y_split).data)

print(f"train loss: {split_loss(Xtr,  Ytr):.4f}")
print(f"dev   loss: {split_loss(Xdev, Ydev):.4f}")
print(f"test  loss: {split_loss(Xte,  Yte):.4f}")

train loss: 2.2570
dev   loss: 2.2641
test  loss: 2.2971


In [21]:
# Evaluate on each split , lr=0.001
def split_loss(X_split, Y_split):
    emb = C[X_split].reshape(X_split.shape[0], -1)
    h = wt.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return float(cross_entropy_loss(logits, Y_split).data)

print(f"train loss: {split_loss(Xtr,  Ytr):.4f}")
print(f"dev   loss: {split_loss(Xdev, Ydev):.4f}")
print(f"test  loss: {split_loss(Xte,  Yte):.4f}")

train loss: 2.2386
dev   loss: 2.2463
test  loss: 2.2780


In [39]:
# Evaluate on each split , lr=0.0001
def split_loss(X_split, Y_split):
    emb = C[X_split].reshape(X_split.shape[0], -1)
    h = wt.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return float(cross_entropy_loss(logits, Y_split).data)

print(f"train loss: {split_loss(Xtr,  Ytr):.4f}")
print(f"dev   loss: {split_loss(Xdev, Ydev):.4f}")
print(f"test  loss: {split_loss(Xte,  Yte):.4f}")

train loss: 2.3318
dev   loss: 2.3363
test  loss: 2.3647


In [40]:
# Sampling from the trained model
# Start with context "..." (all zeros). At each step:
#   1. look up the 3-char context in the embedding table  -> (3, 2)
#   2. flatten to (1, 6) and run the MLP                   -> logits (1, 27)
#   3. softmax -> sample next char index
#   4. slide the context window and append to output
# Stop the word when we sample '.' (index 0), then start the next word.
np.random.seed(4)

for _ in range(10):
    out = []
    context = [0] * block_size

    while True:
        emb = C.data[context]                              # (3, 2)
        emb_wt = wt(emb.reshape(1, -1))                    # (1, 6)
        h = wt.tanh(emb_wt @ W1 + b1)                      # (1, 100)
        logits = h @ W2 + b2                               # (1, 27)

        counts = logits.exp()
        probs = counts / counts.sum(axis=1, keepdims=True) # (1, 27)

        ix = np.random.choice(27, p=probs.data.ravel())
        context = context[1:] + [ix]
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))

yhynn.
zahin.
syci.
iay.
xir.
anandey.
adillen.
midden.
ketaide.
tesholelnon.
